# MCP Fundamentals — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-3/mcp-fundamentals.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-3/mcp-fundamentals.ipynb)

Companion to: [MCP Fundamentals](https://anacodicai.org/learn/ai-agents/6-3/mcp-fundamentals)


## What you'll build

The **real `mcp` library** (the Model Context Protocol SDK) — not a
simulation. MCP is a pure tool/resource protocol: it defines how a client
discovers and calls tools on a server, with **no LLM involved at all**. That
makes it the one framework in this series that needs zero mocking. Same
trip-planning theme as the other notebooks here: a server exposing flight
prices and a baggage-policy lookup.

1. A real `FastMCP` server exposing a tool and a resource.
2. A real `ClientSession` connected to it, discovering what's available.
3. The client calling a tool — and calling a tool that doesn't exist, to see
   how MCP surfaces that as a structured error rather than a crash.

**No API key needed, no LLM at all.** `pip install mcp` if running locally.
The client and server run in the same process here, wired together over an
in-memory transport — the exact test utility MCP's own test suite uses — so
there's no subprocess or network hop to manage inside a notebook. Everything
about the *protocol* is identical to a server running over stdio or SSE.

### Setup

MCP logs an INFO line for every request it processes — useful in production,
noisy in a notebook. Turning it down to `WARNING` keeps the outputs below
readable without changing anything about how the protocol behaves.

In [1]:
# %pip install -q mcp
import asyncio
import logging

logging.getLogger("mcp").setLevel(logging.WARNING)

from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import create_connected_server_and_client_session

print("real mcp SDK imported")

real mcp SDK imported


## Step 1 — a real MCP server with a tool and a resource

`@server.tool()` and `@server.resource(...)` are the real FastMCP decorators
— they register these Python functions so MCP can describe them to any
client that connects, in the protocol's own schema (name, description,
argument types), not just as opaque Python callables.

In [2]:
server = FastMCP("trip-planning-tools")

FLIGHT_PRICES = {
    "boston-barcelona": "$420 per person, round trip",
    "boston-lisbon": "$395 per person, round trip",
}

@server.tool()
def check_flight_price(route: str) -> str:
    """Look up the cached price quote for a given flight route."""
    return FLIGHT_PRICES.get(route, "No cached quote for this route.")

@server.resource("policy://baggage-allowance")
def baggage_allowance() -> str:
    """The airline's baggage policy, exposed as a readable resource."""
    return "One carry-on and one checked bag included; extra bags are $35 each."

print("server 'trip-planning-tools' registered 1 tool and 1 resource")

server 'trip-planning-tools' registered 1 tool and 1 resource


## Step 2 — connect a client, and ask what's available

`create_connected_server_and_client_session` wires a real `ClientSession` to
the server over an in-memory stream pair — the same `ClientSession` class a
production MCP client (an IDE, a chat app, an agent framework) would use
over stdio or HTTP+SSE. Nothing about `check_flight_price` or the resource
was hardcoded into the client below — it asked the server, over the real
protocol, and got back a structured description. That's the actual
mechanism that lets one agent framework talk to MCP servers it has never
seen before.

In [3]:
async def list_everything():
    async with create_connected_server_and_client_session(server._mcp_server) as client:
        tools = await client.list_tools()
        resources = await client.list_resources()
        return tools, resources

tools_result, resources_result = await list_everything()
print("tools the client discovered:   ", [t.name for t in tools_result.tools])
print("resources the client discovered:", [str(r.uri) for r in resources_result.resources])

assert tools_result.tools[0].name == "check_flight_price"
print("\nassert passed: the client discovered the tool purely via the protocol")

tools the client discovered:    ['check_flight_price']
resources the client discovered: ['policy://baggage-allowance']

assert passed: the client discovered the tool purely via the protocol


## Step 3 — the client calls a tool, and reads a resource

In [4]:
async def call_and_read():
    async with create_connected_server_and_client_session(server._mcp_server) as client:
        tool_result = await client.call_tool(
            "check_flight_price", {"route": "boston-barcelona"}
        )
        resource_result = await client.read_resource("policy://baggage-allowance")
        return tool_result, resource_result

tool_result, resource_result = await call_and_read()
print("tool call result:    ", tool_result.content[0].text)
print("resource read result:", resource_result.contents[0].text)

assert "420" in tool_result.content[0].text
print("\nassert passed: the tool returned the cached quote for the requested route")

tool call result:     $420 per person, round trip
resource read result: One carry-on and one checked bag included; extra bags are $35 each.

assert passed: the tool returned the cached quote for the requested route


## Step 4 — calling a tool that doesn't exist

This is the part worth noticing: MCP doesn't raise a Python exception for an
unknown tool name. It returns a normal-looking result with `isError=True` —
a structured protocol-level error a client is expected to check for, the
same way an HTTP client checks a status code instead of only catching
exceptions.

In [5]:
async def call_unknown_tool():
    async with create_connected_server_and_client_session(server._mcp_server) as client:
        return await client.call_tool("cancel_all_bookings", {})

unknown_result = await call_unknown_tool()
print("isError:", unknown_result.isError)
print("content:", unknown_result.content[0].text)

assert unknown_result.isError is True
print("\nassert passed: an unknown tool call came back as a structured error, not a crash")

[08/10/26 08:37:21] WARNING  Tool 'cancel_all_bookings' not listed, no validation will be performed   ]8;id=16000592;file:///usr3/graduate/rashan/.local/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=16000593;file:///usr3/graduate/rashan/.local/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#494\494]8;;\

isError: True
content: Unknown tool: cancel_all_bookings

assert passed: an unknown tool call came back as a structured error, not a crash


## Try it yourself

- Add a second tool, `check_hotel_availability(city: str, dates: str) ->
  str`, and call `client.list_tools()` again — see both tools described
  with their own argument schemas, generated from the Python function
  signatures you wrote.
- Call `check_flight_price` with a route that isn't in `FLIGHT_PRICES` (e.g.
  `"boston-tokyo"`) — notice this one does NOT set `isError=True`, because
  from MCP's point of view the tool call itself succeeded; returning "No
  cached quote for this route." is just this tool's own logic, not a
  protocol failure. Compare that against Step 4's actually-unknown tool.
- Swap the in-memory transport for `server.run_stdio_async()` and connect a
  separate client process to it over stdin/stdout — the tool/resource code
  above doesn't change at all; only the transport does.